In [3]:
from tensorflow import keras
from keras.datasets import mnist
from keras import models, layers
from keras.utils import to_categorical
import optuna
import matplotlib.pyplot as plt

In [4]:
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
train_images = train_images.reshape((-1, 28, 28, 1)) / 255.0
test_images = test_images.reshape((-1, 28, 28, 1)) / 255.0
train_labels = to_categorical(train_labels)
test_labels = to_categorical(test_labels)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [5]:
# %% Define the objective function for Optuna
def objective(trial):
    # Define hyperparameters to tune
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)
    num_filters_1 = trial.suggest_int("num_filters_1", 16, 64, step=16)
    num_filters_2 = trial.suggest_int("num_filters_2", 32, 128, step=32)
    dense_units = trial.suggest_int("dense_units", 32, 128, step=32)
    
    # Build the model
    model = models.Sequential()
    model.add(layers.Conv2D(num_filters_1, (3, 3), activation='relu', input_shape=(28, 28, 1)))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Conv2D(num_filters_2, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Flatten())
    model.add(layers.Dense(dense_units, activation='relu'))
    model.add(layers.Dense(10, activation='softmax'))
    
    # Compile the model
    model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=learning_rate),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    
    # Train the model
    history = model.fit(train_images, train_labels, epochs=5, batch_size=64, verbose=0)
    
    # Return the validation accuracy
    val_acc = history.history['accuracy'][-1]
    return val_acc

In [7]:
# %% Create an Optuna study and optimize
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)  # Adjust the number of trials as needed

[I 2024-11-11 12:17:13,788] A new study created in memory with name: no-name-fcf54cea-55af-4141-9a2b-414f69577370
C:\Users\chydr\AppData\Local\Temp\ipykernel_19148\3375663022.py:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)
[I 2024-11-11 12:18:38,984] Trial 0 finished with value: 0.9823499917984009 and parameters: {'learning_rate': 0.00014053857957223596, 'num_filters_1': 32, 'num_filters_2': 64, 'dense_units': 96}. Best is trial 0 with value: 0.9823499917984009.
[I 2024-11-11 12:21:24,883] Trial 1 finished with value: 0.9955499768257141 and parameters: {'learning_rate': 0.001413403831098812, 'num_filters_1': 64, 'num_filters_2': 96, 'dense_units': 96}. Best is trial 1 with value: 0.9955499768257141.
[I 2024-11-11 12:23:37,106] Trial 2 finished with value:

In [8]:
# %% Display the best parameters and best accuracy achieved
print("Best Parameters:", study.best_params)
print("Best Validation Accuracy:", study.best_value)

Best Parameters: {'learning_rate': 0.001413403831098812, 'num_filters_1': 64, 'num_filters_2': 96, 'dense_units': 96}
Best Validation Accuracy: 0.9955499768257141


In [10]:
# %% Plotting the optimization history
optuna.visualization.plot_optimization_history(study)
plt.show()

ImportError: Tried to import 'plotly' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'plotly'.